<a href="https://colab.research.google.com/github/hanbiphyun/ESSA_YB/blob/main/ESAA_OB_1%EC%A1%B0_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google import colab
colab.drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from scipy.sparse import hstack
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

In [4]:
# ==========================================
# 1. NLTK 리소스 다운로드 (최초 1회만 실행)
# ==========================================
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('universal_tagset')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
train = pd.read_csv('/content/drive/MyDrive/ESSA/novel writer classitication/train.csv')
test = pd.read_csv('/content/drive/MyDrive/ESSA/novel writer classitication/test_x.csv')

In [6]:
# ==========================================
# 2. 텍스트 정제 및 기본 전처리 함수
# ==========================================
def get_wordnet_pos(treebank_tag):
    """NLTK 품사 태그를 WordNet 품사 태그로 변환"""
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def apply_lemmatization(text):
    """품사를 고려한 표제어 추출(Lemmatization)"""
    words = text.split()
    pos_tags = pos_tag(words)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    return " ".join(lemmatized_words)

# 문장 길이 분포 확인 및 max_length 설정
train['word_len'] = train['text'].apply(lambda x: len(x.split()))
quantiles = train['word_len'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])
print("📊 학습 데이터 단어 길이(word_len) 분포:\n", quantiles)

# 99% 혹은 95% 데이터를 포괄하는 길이를 max_length로 지정
max_length = int(quantiles[0.99])
print(f"✅ 권장 max_length 지정: {max_length}")

# 표제어 추출 적용
print("⏳ 표제어 추출(Lemmatization) 진행 중...")
train['text_lemma'] = train['text'].apply(apply_lemmatization)
test['text_lemma'] = test['text'].apply(apply_lemmatization)

📊 학습 데이터 단어 길이(word_len) 분포:
 0.50     22.0
0.75     50.0
0.90    111.0
0.95    158.1
0.99    242.0
Name: word_len, dtype: float64
✅ 권장 max_length 지정: 242
⏳ 표제어 추출(Lemmatization) 진행 중...


In [ ]:
# ==========================================
# 3. 수치형 피처 엔지니어링 (Feature Engineering)
# ==========================================
def extract_features(df):
    """텍스트 기반 수치형 피처 추출 (Train/Test 동일 적용을 위해 함수화)"""
    df_features = pd.DataFrame(index=df.index)

    # 기초 통계
    df_features['len'] = df['text'].apply(len)
    df_features['words'] = df['text'].apply(lambda x: len(x.split()))

    # 구두점 및 특수문자
    df_features['comma_cnt'] = df['text'].apply(lambda x: x.count(','))
    df_features['dot_cnt'] = df['text'].apply(lambda x: x.count('.'))
    df_features['ques_cnt'] = df['text'].apply(lambda x: x.count('?'))
    df_features['excl_cnt'] = df['text'].apply(lambda x: x.count('!'))
    df_features['quote_cnt'] = df['text'].apply(lambda x: x.count('"') + x.count("'"))
    df_features['special_punc'] = df['text'].apply(lambda x: x.count('?"') + x.count('!"') + x.count('."'))

    # 비율 관련 피처
    df_features['comma_ratio'] = df_features['comma_cnt'] / (df_features['len'] + 1)
    df_features['word_len_avg'] = df_features['len'] / (df_features['words'] + 1)

    # 어휘 다양성 및 스톱워드 밀도
    df_features['ttr'] = df.apply(lambda x: len(set(x['text'].split())) / (len(x['text'].split()) + 1), axis=1)
    df_features['stop_density'] = df['text'].apply(lambda x: len([w for w in x.split() if w.lower() in stop_words]) / (len(x.split()) + 1))

    # 대명사 및 관사 비중
    pronouns = ['he', 'she', 'it', 'they', 'him', 'her', 'them', 'his', 'hers', 'their']
    articles = ['the', 'a', 'an']
    df_features['pronoun_ratio'] = df['text'].apply(lambda x: len([w for w in x.split() if w.lower() in pronouns]) / (len(x.split()) + 1))
    df_features['article_ratio'] = df['text'].apply(lambda x: len([w for w in x.split() if w.lower() in articles]) / (len(x.split()) + 1))

    # 품사 카운팅 (VB, JJ, NN)
    def get_pos_counts(text):
        tokens = word_tokenize(text)
        tags = pos_tag(tokens)
        counts = {'VB': 0, 'JJ': 0, 'NN': 0}
        for word, tag in tags:
            if tag.startswith('VB'): counts['VB'] += 1
            elif tag.startswith('JJ'): counts['JJ'] += 1
            elif tag.startswith('NN'): counts['NN'] += 1
        return pd.Series(counts)

    pos_df = df['text'].apply(get_pos_counts)
    df_features = pd.concat([df_features, pos_df], axis=1)

    return df_features.fillna(0)

print("⏳ 수치형 피처 추출 중...")
train_num_features = extract_features(train)
test_num_features = extract_features(test)

# 수치형 피처 스케일링
scaler = StandardScaler()
train_num_scaled = scaler.fit_transform(train_num_features)
test_num_scaled = scaler.transform(test_num_features)

⏳ 수치형 피처 추출 중...


In [ ]:
# ==========================================
# 4. TF-IDF 벡터화 및 데이터 병합
# ==========================================
# (1) Word TF-IDF (Lemmatization된 텍스트 사용 권장)
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000)
X_train_word = tfidf_word.fit_transform(train['text_lemma'])
X_test_word = tfidf_word.transform(test['text_lemma'])

# (2) Char TF-IDF (글자 단위 습관 포착, 원본 텍스트 사용)
tfidf_char = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=3, max_features=10000)
X_train_char = tfidf_char.fit_transform(train['text'])
X_test_char = tfidf_char.transform(test['text'])

# 최종 데이터 셋 (TF-IDF + 수치형 피처 결합)
X_final = hstack([X_train_word, X_train_char, train_num_scaled])
X_test_final = hstack([X_test_word, X_test_char, test_num_scaled])
y = train['author']

print(f"✅ 학습 데이터 최종 피처 개수: {X_final.shape[1]}")
print(f"✅ 테스트 데이터 최종 피처 개수: {X_test_final.shape[1]}")

✅ 학습 데이터 최종 피처 개수: 25017
✅ 테스트 데이터 최종 피처 개수: 25017


In [ ]:
# ==========================================
# 5. 딥러닝 고도화: BERT 3종 성능 비교 (Experiment)
# ==========================================
import torch
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

print("\n🚀 [Step 1] BERT 3종 성능 비교 실험 시작...")

# 🌟 중요: 클래스 불균형을 해결하기 위해 반드시 stratify=y 를 적용하여 8:2 분할!
# (팀원분들이 만들어둔 text_lemma 사용)
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train['author'])

train_dataset = Dataset.from_pandas(train_df[['text_lemma', 'author']])
val_dataset = Dataset.from_pandas(val_df[['text_lemma', 'author']])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    return {"log_loss": log_loss(labels, probs, labels=[0, 1, 2, 3, 4])}

# 테스트할 3가지 모델 리스트
models_to_test = [
    "albert-base-v2",     # 가볍고 빠른 모델
    "bert-base-uncased",  # 스탠다드 모델
    "roberta-base"        # 텍스트 분류 최강자
]

model_results = {}

for model_name in models_to_test:
    print(f"\n{'='*50}")
    print(f"🔥 학습 진행 중인 모델: {model_name}")
    print(f"{'='*50}")

    # 1. 토크나이저 및 모델 로드
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model_bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

    # 2. 토큰화 진행
    def tokenize_function(examples):
        return tokenizer(examples["text_lemma"], padding="max_length", truncation=True, max_length=256)

    tokenized_train = train_dataset.map(tokenize_function, batched=True).rename_column("author", "labels")
    tokenized_val = val_dataset.map(tokenize_function, batched=True).rename_column("author", "labels")

    # 3. 학습 설정 (비교를 위해 2에폭만 빠르게 진행)
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.replace('/', '_')}",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16, # VRAM 부족 시 8로 수정
        per_device_eval_batch_size=32,
        num_train_epochs=2,             # 성능 랭킹만 보기 위해 2회로 축소
        weight_decay=0.01,
        fp16=True,                      # GPU 가속
        save_strategy="no",             # 용량 절약을 위해 모델 저장 안 함
        report_to="none"                # 불필요한 로그 끄기
    )

    trainer = Trainer(
        model=model_bert,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    # 4. 학습 및 평가
    trainer.train()
    eval_result = trainer.evaluate()

    # 5. 결과 기록
    model_results[model_name] = eval_result['eval_log_loss']
    print(f"🎯 {model_name} 검증 LogLoss: {eval_result['eval_log_loss']:.5f}")

    # 6. GPU 메모리 초기화 (다음 모델 학습을 위해 필수)
    del model_bert, trainer, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n🏆 모델 비교 최종 결과 (LogLoss가 낮을수록 좋음)")
for m, score in sorted(model_results.items(), key=lambda item: item[1]):
    print(f"- {m}: {score:.5f}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

print("🚀 [Step 1] bert-base-uncased 모델 학습 및 평가 시작...")

# 1. 데이터 분할 (앙상블 검증을 위해 8:2 유지, 불균형 고려 stratify=y)
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train['author'])

train_dataset = Dataset.from_pandas(train_df[['text_lemma', 'author']].fillna(""))
val_dataset = Dataset.from_pandas(val_df[['text_lemma', 'author']].fillna(""))
test_dataset = Dataset.from_pandas(test[['text_lemma']].fillna(""))

# 2. 모델 및 토크나이저 로드
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

def tokenize_func(examples):
    return tokenizer(examples["text_lemma"], padding="max_length", truncation=True, max_length=256)

print("⏳ 토큰화 진행 중...")
tokenized_train = train_dataset.map(tokenize_func, batched=True).rename_column("author", "labels")
tokenized_val = val_dataset.map(tokenize_func, batched=True).rename_column("author", "labels")
tokenized_test = test_dataset.map(tokenize_func, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()
    return {"log_loss": log_loss(labels, probs, labels=[0, 1, 2, 3, 4])}

# 3. 학습 설정 및 실행
training_args = TrainingArguments(
    output_dir="./results_bert_uncased",
    eval_strategy="epoch",  # 최신 버전에 맞게 수정됨
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="log_loss",
    greater_is_better=False,
    save_strategy="epoch",
)

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

# 4. 검증 데이터 및 테스트 데이터 예측 (앙상블용 보관)
print("\n🎯 검증 및 테스트 데이터 예측 중...")
val_preds_output = trainer.predict(tokenized_val)
bert_val_preds = torch.nn.functional.softmax(torch.tensor(val_preds_output.predictions), dim=-1).numpy()

test_preds_output = trainer.predict(tokenized_test)
bert_test_preds = torch.nn.functional.softmax(torch.tensor(test_preds_output.predictions), dim=-1).numpy()

# 5. BERT 단일 모델 결과 저장
submission = pd.read_csv('/content/drive/MyDrive/ESSA/novel writer classitication/sample_submission.csv')
submission[['0', '1', '2', '3', '4']] = bert_test_preds

bert_submission_path = '/content/drive/MyDrive/ESSA/novel writer classitication/submission_bert_uncased_only.csv'
submission.to_csv(bert_submission_path, index=False)
print(f"🎉 BERT 단일 제출 파일 저장 완료:\n{bert_submission_path}")